# 00 — Setup
Mount Drive, download RealWaste into the shared folder, check the repo.

In [6]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import torch, os, platform
print('Python platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Python platform: Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [9]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Deep Learning - RealWaste')  # CHANGE if needed
DATASET_ROOT = DRIVE_ROOT / '01_Dataset' / 'RealWaste'
SPLIT_DIR = DRIVE_ROOT / '02_Splits'
CHECKPOINT_DIR = DRIVE_ROOT / '03_Checkpoints'
RESULT_DIR = DRIVE_ROOT / '04_Results'

for p in [SPLIT_DIR, CHECKPOINT_DIR, RESULT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('DRIVE_ROOT:', DRIVE_ROOT)
print('DATASET_ROOT exists:', DATASET_ROOT.exists())
print('SPLIT_DIR:', SPLIT_DIR)
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)
print('RESULT_DIR:', RESULT_DIR)


DRIVE_ROOT: /content/drive/MyDrive/Deep Learning - RealWaste
DATASET_ROOT exists: True
SPLIT_DIR: /content/drive/MyDrive/Deep Learning - RealWaste/02_Splits
CHECKPOINT_DIR: /content/drive/MyDrive/Deep Learning - RealWaste/03_Checkpoints
RESULT_DIR: /content/drive/MyDrive/Deep Learning - RealWaste/04_Results


In [11]:
!pip -q install kagglehub

In [ ]:
import kagglehub

EXPECTED_IMAGES = 4752  # verified by 01_dataset_audit (DECISION_LOG D006)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

existing_images = sum(
    1 for p in DATASET_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
) if DATASET_ROOT.exists() else 0

if existing_images > EXPECTED_IMAGES:
    raise RuntimeError(
        f"{existing_images} images in {DATASET_ROOT}, expected {EXPECTED_IMAGES}. Investigate before continuing."
    )

NEED_COPY = existing_images < EXPECTED_IMAGES
if NEED_COPY:
    # kagglehub returns a str; wrap it in Path so the "/" joins in the next cell work.
    download_path = Path(kagglehub.dataset_download("joebeachcapital/realwaste"))
    print("Downloaded to:", download_path)
else:
    print(f"Dataset already present ({existing_images} images) - skipping download and copy.")

In [ ]:
import shutil

TARGET_DATASET = DATASET_ROOT

if NEED_COPY:
    SOURCE_DATASET = download_path / "realwaste-main" / "RealWaste"
    print("Source:", SOURCE_DATASET)
    print("Target:", TARGET_DATASET)
    assert SOURCE_DATASET.exists(), "Source dataset not found."

    # Copy dataset into Drive
    shutil.copytree(SOURCE_DATASET, TARGET_DATASET, dirs_exist_ok=True)
    print("Copy completed.")
else:
    print("Using existing dataset at", TARGET_DATASET)

In [16]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

class_dirs = sorted([
    p for p in TARGET_DATASET.iterdir()
    if p.is_dir()
])

print("Classes found:", len(class_dirs))

total_images = 0

for class_dir in class_dirs:
    images = [
        p for p in class_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    print(f"{class_dir.name:25} {len(images)}")
    total_images += len(images)

print("\nTotal image files:", total_images)

Classes found: 9
Cardboard                 461
Food Organics             411
Glass                     420
Metal                     790
Miscellaneous Trash       495
Paper                     500
Plastic                   921
Textile Trash             318
Vegetation                436

Total image files: 4752
